### RETREIVAL AGENT

User query ---> Query Agent ---> Structured Query ---> Retrieval Agent ---> FAISS ---> Top-K Products

In [1]:
import math
import re

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from pydantic import BaseModel, Field

load_dotenv()


C:\Users\srush\AppData\Local\Temp\ipykernel_37540\4261633159.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

In [2]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [3]:
#loading FAISS Database

vector_db = FAISS.load_local(
    "../Data/Cleaned/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

recieving a structured query

In [4]:
class ProductQuery(BaseModel):

    product_type: str

    brand: str | None = None

    budget: float | None = None

    features: list[str] = Field(
        default_factory=list
    )
    

sample query

In [5]:
query = ProductQuery(
    product_type="headset",
    brand="Apple",
    budget=30000,
    features=[
        "noise cancellation",
        "wireless"
    ]
)

query

ProductQuery(product_type='headset', brand='Apple', budget=30000.0, features=['noise cancellation', 'wireless'])

Building search query

In [6]:
ALIASES = {
    "phone": "phone smartphone mobile iphone galaxy pixel",
    "headset": "headset headphones earphones earbuds airpods bluetooth wireless",
    "laptop": "laptop notebook macbook"
}

def build_search_query(query: ProductQuery) -> str:

    aliases = ALIASES.get(
        query.product_type.lower(),
        query.product_type
    )

    feature_text = " ".join(query.features)

    brand_text = query.brand or ""

    return f"""
Brand: {brand_text}
Product: {aliases}
Features: {feature_text}
""".strip()

In [7]:
search_query = build_search_query(query)

print(search_query)

Brand: Apple
Product: headset headphones earphones earbuds airpods bluetooth wireless
Features: noise cancellation wireless


In [8]:
# #filteringfunction:


# def filter_products(results, query):

#     filtered = []

#     for doc, score in results:

#         title = doc.metadata.get("title", "").lower()

#         category = doc.metadata.get("categories", "").lower()

#         product_text = f"{title} {category}"

#         # -----------------------
#         # Brand Filter
#         # -----------------------

#         if query.brand:

#             if query.brand.lower() not in product_text:
#                 continue

#         # -----------------------
#         # Product Type Filter
#         # -----------------------

#         if query.product_type:

#             if query.product_type.lower() not in product_text:
#                 continue

#         filtered.append((doc, score))

#     return filtered



In [9]:
KNOWN_BRANDS = [
    "apple",
    "samsung",
    "google",
    "oneplus",
    "xiaomi",
    "realme",
    "oppo",
    "vivo",
    "motorola",
    "nokia",
    "sony",
    "asus",
    "jbl",
    "boat",
    "beats",
    "bose",
    "sennheiser",
    "anker"
]

In [10]:
def extract_brand(title: str) -> str | None:

    title = str(title).lower()

    for brand in KNOWN_BRANDS:

        pattern = rf"\b{re.escape(brand)}\b"

        if re.search(pattern, title):
            return brand

    return None

In [11]:
PRODUCT_TYPES = {

    "phone": [
        "phone",
        "smartphone",
        "mobile",
        "iphone",
        "galaxy",
        "pixel"
    ],

    "headset": [
        "headset",
        "headphone",
        "headphones",
        "earphone",
        "earphones",
        "earbud",
        "earbuds",
        "airpods"
    ],

    "laptop": [
        "laptop",
        "notebook",
        "macbook"
    ]
}

In [12]:
NEGATIVE_KEYWORDS = {

    "phone": [
        "case",
        "cover",
        "screen protector",
        "protector",
        "charger",
        "adapter",
        "cable",
        "replacement battery",
        "battery cover",
        "remote control",
        "camera shutter",
        "selfie remote",
        "selfie stick",
        "tripod",
        "mount",
        "holder",
        "stand",
        "lens",
        "stylus",
        "screen",
        "parts",
        "accessory",
        "accessories"
    ],

    "headset": [
        "adapter",
        "jack",
        "connector",
        "converter",
        "replacement cable",
        "charging cable",
        "case",
        "cover",
        "holder",
        "stand",
        "ear pads",
        "ear cushions"
    ],

    "laptop": [
        "charger",
        "keyboard cover",
        "screen protector",
        "case",
        "battery",
        "adapter",
        "sleeve",
        "stand"
    ]
}

In [13]:
def is_actual_product(
    title: str,
    product_type: str
) -> bool:

    title = str(title).lower()
    product_type = product_type.lower()

    if product_type == "phone":

        strong_phone_terms = [
            "smartphone",
            "cell phone",
            "cellphone",
            "mobile phone",
            "android phone",
            "iphone",
            "galaxy s",
            "galaxy a",
            "galaxy note",
            "galaxy mega",
            "galaxy rugby",
            "pixel"
        ]

        return any(
            term in title
            for term in strong_phone_terms
        )

    if product_type == "headset":

        strong_headset_terms = [
            "headset",
            "headphone",
            "headphones",
            "earphone",
            "earphones",
            "earbud",
            "earbuds",
            "airpods"
        ]

        return any(
            term in title
            for term in strong_headset_terms
        )

    if product_type == "laptop":

        strong_laptop_terms = [
            "laptop",
            "notebook",
            "macbook"
        ]

        return any(
            term in title
            for term in strong_laptop_terms
        )

    return product_type in title

In [14]:
def filter_products(
    results,
    query: ProductQuery
):

    filtered = []

    for doc, score in results:

        title = str(
            doc.metadata.get("title", "")
        ).lower()

        category = str(
            doc.metadata.get("categories", "")
        ).lower()

        text = f"{title} {category}"

        # ---------------------------------
        # Brand filter
        # ---------------------------------

        if query.brand:

            predicted_brand = extract_brand(title)

            if predicted_brand != query.brand.lower():
                continue

        # ---------------------------------
        # Product-type keyword filter
        # ---------------------------------

        positive_words = PRODUCT_TYPES.get(
            query.product_type.lower(),
            [query.product_type.lower()]
        )

        if not any(
            word in text
            for word in positive_words
        ):
            continue

        # ---------------------------------
        # Strong product validation
        # ---------------------------------

        if not is_actual_product(
            title,
            query.product_type
        ):
            continue

        # ---------------------------------
        # Accessory exclusion
        # ---------------------------------

        negative_words = NEGATIVE_KEYWORDS.get(
            query.product_type.lower(),
            []
        )

        if any(
            word in title
            for word in negative_words
        ):
            continue

        # ---------------------------------
        # Minimum rating
        # ---------------------------------

        try:
            rating = float(
                doc.metadata.get(
                    "average_rating",
                    0
                )
            )

        except (TypeError, ValueError):
            rating = 0.0

        if rating < 3.5:
            continue

        if query.budget is not None:

            try:
                price = float(
                    doc.metadata.get("price")
                )

            except (TypeError, ValueError):
                price = None

            if price is None:
                continue

            if price > query.budget:
                continue

        filtered.append(
            (doc, score)
        )

    return filtered

In [15]:
def remove_duplicates(results):

    seen = set()
    unique = []

    for doc, score in results:

        asin = doc.metadata.get(
            "parent_asin"
        )

        if asin is None:
            continue

        if asin in seen:
            continue

        seen.add(asin)

        unique.append(
            (doc, score)
        )

    return unique

In [16]:
class RetrievalAgent:

    def __init__(self, vector_db):

        self.vector_db = vector_db

    def retrieve(
        self,
        query: ProductQuery,
        k: int = 50,
        top_n: int = 10
    ):

        search_query = build_search_query(query)

        results = (
            self.vector_db
            .similarity_search_with_score(
                search_query,
                k=k
            )
        )

        results = filter_products(
            results,
            query
        )

        results = remove_duplicates(
            results
        )

        scored_results = []

        for doc, similarity_distance in results:

            similarity = 1 / (
                1 + similarity_distance
            )

            try:
                rating = float(
                    doc.metadata.get(
                        "average_rating",
                        0
                    )
                )

            except (TypeError, ValueError):
                rating = 0.0

            try:
                rating_number = float(
                    doc.metadata.get(
                        "rating_number",
                        0
                    )
                )

            except (TypeError, ValueError):
                rating_number = 0.0

            popularity = min(
                math.log10(
                    rating_number + 1
                ) / 5,
                1
            )

            retrieval_score = (
                similarity * 0.65
                + (rating / 5) * 0.20
                + popularity * 0.15
            )

            doc.metadata["similarity"] = similarity

            doc.metadata[
                "retrieval_score"
            ] = retrieval_score

            doc.metadata["brand"] = (
                extract_brand(
                    doc.metadata.get(
                        "title",
                        ""
                    )
                )
            )

            doc.metadata[
                "product_type"
            ] = query.product_type

            doc.metadata[
                "requested_features"
            ] = query.features

            doc.metadata[
                "budget"
            ] = query.budget

            scored_results.append(
                (doc, retrieval_score)
            )

        scored_results.sort(
            key=lambda item: item[1],
            reverse=True
        )

        return scored_results[:top_n]

In [17]:
retrieval_agent = RetrievalAgent(
    vector_db
)

In [18]:
results = retrieval_agent.retrieve(
    query,
    k=100,
    top_n=10
)

In [19]:
print(
    "Number of products retrieved:",
    len(results)
)

for rank, (doc, score) in enumerate(
    results,
    start=1
):

    print("=" * 80)

    print("Rank:", rank)

    print(
        "Title:",
        doc.metadata.get("title")
    )

    print(
        "Brand:",
        doc.metadata.get("brand")
    )

    print(
        "Rating:",
        doc.metadata.get(
            "average_rating"
        )
    )

    print(
        "Rating Count:",
        doc.metadata.get(
            "rating_number"
        )
    )

    print(
        "Price:",
        doc.metadata.get("price")
    )

    print(
        "Similarity:",
        round(
            doc.metadata.get(
                "similarity",
                0
            ),
            4
        )
    )

    print(
        "Retrieval Score:",
        round(score, 4)
    )

    print()

Number of products retrieved: 4
Rank: 1
Title: vodbov Bluetooth Headset, Wireless earpiece for Business/Sport/Driver Noise Cancelling Mic Black Bluetooth Headphones Compatible with Apple Android Cell Phones,MacBook and Compatible with Alexa
Brand: apple
Rating: 3.9
Rating Count: 158
Price: nan
Similarity: 0.5686
Retrieval Score: 0.5916

Rank: 2
Title: Travel 3 in 1 Magsafe Wireless Charger, Foldable Wireless Charging Station for Apple, Wireless Charging Pad Compatible with iPhone 14 13 12 11/Pro/XS/XR,AirPods 3/2/Pro, iWatch 7/6/5/4/3/2
Brand: apple
Rating: 4.6
Rating Count: 124
Price: 25.99
Similarity: 0.4874
Retrieval Score: 0.5637

Rank: 3
Title: Foldable Wireless Charging Station for Multiple Devices Apple, 3 in 1 Wireless Charger for iPhone 13/12/11 Pro Max/X/Xs Max/8/8 Plus,for iWatch 7/6/5/4/3/2/se, AirPods 3/2/pro (Switchable Light)
Brand: apple
Rating: 4.1
Rating Count: 545
Price: 21.99
Similarity: 0.4848
Retrieval Score: 0.5612

Rank: 4
Title: JNTworld White Foldable DJ Overh